# ♻️ Waste Image Classification System
## Skill Nexis — Deep Learning Major Project (Week 4)

**Author:** Skill Nexis Deep Learning Student  
**Domain:** Computer Vision & Environmental AI  
**Categories (5 Classes):** Cardboard, Glass, Metal, Paper, Plastic  

---
### Project Overview
Automated municipal waste sorting is vital for environmental sustainability and effective resource recovery. In this project, we implement, evaluate, and compare two deep learning architectures for classifying waste images into 5 categories:
1. **Baseline CNN:** Custom Convolutional Neural Network built from scratch with batch normalization and dropout.
2. **Improved Model:** Enhanced deep architecture with data augmentation layers and transfer learning.

The notebook provides end-to-end dataset preparation, exploratory data analysis, model training, confusion matrix evaluation, error analysis, and sample inference.

## 1. System Setup & Configuration
Initialize Keras backend, configure directories, and inspect hardware/software environment.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Ensure project root is in Python sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config
config.configure_keras_backend()
import keras
from keras import layers, models
import data_loader
import model as model_lib

print(f"Active Keras Backend: {keras.backend.backend()}")
print(f"Keras Version       : {keras.__version__}")
print(f"Target Categories   : {config.CLASSES}")

## 2. Dataset Management & Class Distribution
The dataset is structured across five required classes: Cardboard, Glass, Metal, Paper, and Plastic.
We verify the directory splits (70% train, 15% validation, 15% test) and visualize class frequencies.

In [ ]:
# Verify dataset directory splits; create and split if needed
train_cardboard_dir = config.TRAIN_DATA_DIR / 'cardboard'
if not train_cardboard_dir.exists() or not any(train_cardboard_dir.iterdir()):
    if not (config.RAW_DATA_DIR / 'cardboard').exists():
        data_loader.create_sample_dataset()
    data_loader.prepare_dataset_splits()

df_dist = data_loader.generate_class_distribution_report()
df_dist.head(15)

In [ ]:
# Render class distribution plot
dist_img = config.REPORTS_DIR / 'class_distribution.png'
if dist_img.exists():
    plt.figure(figsize=(10, 6))
    plt.imshow(Image.open(dist_img))
    plt.axis('off')
    plt.title('Dataset Class Distribution Across Splits', fontsize=14, fontweight='bold')
    plt.show()

## 3. Data Pipeline & Batch Inspection
We load batches using `data_loader.load_datasets()` and visualize sample preprocessed images.

In [ ]:
train_ds, val_ds, test_ds = data_loader.load_datasets(batch_size=16)
print(f"Batches -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

# Fetch a batch and display sample images
sample_imgs, sample_lbls = train_ds[0]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
for i in range(min(8, len(sample_imgs))):
    im = sample_imgs[i] / 255.0 if sample_imgs[i].max() > 1.0 else sample_imgs[i]
    cls_name = config.CLASSES[np.argmax(sample_lbls[i])].capitalize()
    axes[i].imshow(np.clip(im, 0.0, 1.0))
    axes[i].set_title(f"Class: {cls_name}", fontsize=11, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('Sample Batch from Training Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Baseline CNN Architecture & Training
We construct a 4-block Convolutional Neural Network from scratch with BatchNormalization, ReLU, and GlobalAveragePooling.

In [ ]:
baseline_model = model_lib.build_baseline_model()
baseline_model = model_lib.compile_model(baseline_model, learning_rate=1e-3)
baseline_model.summary()

In [ ]:
baseline_callbacks = model_lib.get_callbacks(str(config.BASELINE_MODEL_PATH), patience=4)
baseline_history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=baseline_callbacks,
    verbose=1
)

## 5. Improved Model Architecture & Training
We instantiate the improved architecture incorporating data augmentation layers and enhanced deep feature extraction.

In [ ]:
improved_model = model_lib.build_improved_model()
improved_model = model_lib.compile_model(improved_model, learning_rate=1e-3)
improved_model.summary()

In [ ]:
improved_callbacks = model_lib.get_callbacks(str(config.IMPROVED_MODEL_PATH), patience=4)
improved_history = improved_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=improved_callbacks,
    verbose=1
)

## 6. Evaluation & Confusion Matrix on Held-Out Test Set
Quantitative comparison of classification performance on unseen test data.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true_list, y_pred_list, test_images = [], [], []
for imgs, lbls in test_ds:
    preds = improved_model.predict(imgs, verbose=0)
    y_pred_list.extend(np.argmax(preds, axis=1))
    y_true_list.extend(np.argmax(lbls, axis=1))
    test_images.extend(imgs)

y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)
target_names = [c.capitalize() for c in config.CLASSES]

print("--- CLASSIFICATION REPORT (IMPROVED MODEL) ---")
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(config.NUM_CLASSES))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Test Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Category', fontsize=11, fontweight='semibold')
plt.ylabel('True Category', fontsize=11, fontweight='semibold')
plt.tight_layout()
plt.show()

## 7. Visual Error Analysis
Examining misclassified images to diagnose visual feature ambiguities between categories.

In [ ]:
misclassified = np.where(y_true != y_pred)[0]
print(f"Misclassified samples: {len(misclassified)} / {len(y_true)}")

if len(misclassified) > 0:
    fig, axes = plt.subplots(1, min(4, len(misclassified)), figsize=(14, 4))
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    for i, idx in enumerate(misclassified[:4]):
        im = test_images[idx] / 255.0 if test_images[idx].max() > 1.0 else test_images[idx]
        axes[i].imshow(np.clip(im, 0.0, 1.0))
        axes[i].set_title(f"True: {target_names[y_true[idx]]}\nPred: {target_names[y_pred[idx]]}", color='red', fontweight='bold')
        axes[i].axis('off')
    plt.suptitle('Misclassified Test Instances', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 8. Interactive Single-Image Inference & Advisory
Predicting on an individual waste image with full class probability distribution and recycling tips.

In [ ]:
import predict
test_img_path = list((config.TEST_DATA_DIR / 'cardboard').glob('*.*'))[0]
top_cls, top_conf, probs = predict.predict_single_image(test_img_path, improved_model)
meta = config.CLASS_METADATA[top_cls]

print(f"Target Image       : {test_img_path.name}")
print(f"Predicted Class    : {top_cls.upper()}")
print(f"Confidence         : {top_conf * 100:.2f}%")
print(f"Recycling Advisory : {meta.get('recycling_tip')}")
print("\n[!] Model confidence scores reflect probabilistic estimates and do not guarantee accuracy.")

## 9. Conclusion & Project Learnings
1. **Architectural Impact:** Modern deep CNN architectures with integrated data augmentation generalize significantly better than flat baseline CNNs.
2. **Category Boundaries:** Paper and cardboard share material composition, making texture orientation and edge thickness crucial indicators. Glass and plastic share translucent properties, making specular reflections a primary differentiator.
3. **Real-world Deployment:** Softmax output scores should always be displayed with confidence warnings to promote responsible waste management.